# Day 4: Travel Planner (Memory-Based Bot)
Implementing a multi-turn conversational agent using LangGraph and LangChain. 
- **Goal:** Retain user preferences (destination, dates, budget) across turns.
- **Tools used:** `MemorySaver` for state checkpointer tracking via a unique `thread_id`.

In [2]:
# Install the required LangGraph and Anthropic integration libraries
!pip install -q langgraph langchain-anthropic --break-system-packages

In [5]:
#### Setup

In [3]:
from dotenv import load_dotenv
import os
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

# Load variables from your .env file
load_dotenv()
 
api_key = os.getenv("KEY")
base_url = os.getenv("BASE_URL")
model_name = os.getenv("MODEL")
 
# Initialize model through your custom lab gateway
llm = ChatAnthropic(
        model=model_name,
        api_key=api_key,
        anthropic_api_url=base_url,
        temperature=0
    )
print("Successfully initialized ChatAnthropic client.")


Successfully initialized ChatAnthropic client.


#### Define state and bot node

In [4]:
# Define the agent state tracking mechanism
class State(TypedDict):
    # add_messages ensures new turns append to the chat history instead of overwriting it
    messages: Annotated[list, add_messages]

# Define the conversational node
def travel_planner_bot(state: State):
    system_prompt = SystemMessage(
        content="You are a precise travel assistant. Your job is to remember user details "
                "(destination, dates, budget) across multiple conversational turns. "
                "When requested to summarize, provide a concise summary matching exactly what they provided."
    )
    # Stitch the tracking history together with system context
    messages = [system_prompt] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}

#### Graph Compilation

In [6]:
# Construct the state workflow graph
workflow = StateGraph(State)

# Add node and map boundaries
workflow.add_node("planner", travel_planner_bot)
workflow.add_edge(START, "planner")
workflow.add_edge("planner", END)

# Configure MemorySaver checkpointer for managing session memory
memory = MemorySaver()

# Compile the final executable app
app = workflow.compile(checkpointer=memory)
print("Graph successfully compiled with persistent memory checkpointer.")

Graph successfully compiled with persistent memory checkpointer.


#### Multiturn Execution Loop

In [8]:
# Configuration containing the critical thread_id session key (file "7444782a-774e-4b23-a839-390217ec6544")
config = {"configurable": {"thread_id": "workspace_session_anthropic_02"}}

# Turn 1: Destination
t1 = app.invoke({"messages": [HumanMessage(content="I am planning a trip to Bali.")]}, config)
print(f"User: I am planning a trip to Bali.\nBot: {t1['messages'][-1].content}\n" + "-"*40)

User: I am planning a trip to Bali.
Bot: I already have your Bali trip details on file! Let me remind you of what we've discussed so far:

**Your Current Plan:**
- **Destination:** Bali
- **Travel month:** December
- **Accommodation preference:** Budget hotels

To help you further, I still need:
- **Specific dates in December** (or duration of stay?)
- **Your overall budget** for the trip
- **Preferred areas** of Bali (Ubud, Seminyak, Canggu, Sanur, etc.)

Would you like to add any of these details, or do you have questions about your December trip to Bali?
----------------------------------------


In [9]:
# Turn 2: Budget Preference
t2 = app.invoke({"messages": [HumanMessage(content="I prefer budget hotels.")]}, config)
print(f"User: I prefer budget hotels.\nBot: {t2['messages'][-1].content}\n" + "-"*40)

User: I prefer budget hotels.
Bot: Got it! I already have that noted—you prefer budget hotels for your Bali trip in December.

To move forward with planning, I still need:

- **Specific dates in December** (or how many days will you stay?)
- **Your overall budget** for the trip
- **Which areas of Bali interest you?** (Ubud, Seminyak, Canggu, Sanur, etc.)

Once you provide these details, I can recommend specific budget-friendly hotels and create a personalized itinerary for you!
----------------------------------------


In [10]:
# Turn 3: Timeline
t3 = app.invoke({"messages": [HumanMessage(content="My trip is in December.")]}, config)
print(f"User: My trip is in December.\nBot: {t3['messages'][-1].content}\n" + "-"*40)

User: My trip is in December.
Bot: I already have that information—you're traveling to Bali in December and prefer budget hotels. 

To complete your travel plan, I still need:

- **Specific dates in December** (or duration—how many days/weeks?)
- **Your overall budget** for the trip
- **Which areas of Bali interest you?** (Ubud, Seminyak, Canggu, Sanur, etc.)

These final details will help me give you the best recommendations for budget hotels and activities!
----------------------------------------


In [11]:
# Turn 4: Summarize
t4 = app.invoke({"messages": [HumanMessage(content="Summarise my travel plan.")]}, config)
print(f"User: Summarise my travel plan.\n\nBot Output:\n{t4['messages'][-1].content}")

User: Summarise my travel plan.

Bot Output:
Here's a summary of your travel plan:

**Destination:** Bali

**Travel Time:** December

**Accommodation Preference:** Budget hotels

**Still needed to complete your plan:**
- Specific dates in December (duration of stay)
- Overall budget for the trip
- Preferred areas of Bali
